# 10 — Agent logic: pressure, forecast bands, allocation decisions

Phase B, built on the closed forecasting result (04, 07–09): the forecast layer is **persistence + a calibrated 90% band**. This notebook turns that into auditable allocation decisions for C_110 at public allocatable providers, then checks the decisions against what actually happened six months later.

**Declared constants** (declared 2026-07-30, *not tuned against any outcome*; pending supervisor ratification):

- Pressure weights **0.2 / 0.6 / 0.2** on (>18-week share, >52-week share, G&A occupancy). Rationale from measured evidence: the >52-week share is the component that exploded post-Covid (0.05% → 4.11%, 81× — see 02/summary), so it carries the largest weight; occupancy adds operational strain context though it has near-zero forecasting importance (07).
- Decision thresholds: high pressure ≥ **0.70**, moderate ≥ **0.40**; growth risk = band-upper ≥ **1.20** × current; stability = band-upper ≤ **1.05** × current; spare capacity = occupancy < **0.80**.

**Design points, forced by arithmetic:**

1. **Components are ranked within each month** (cross-sectional percentile among the cohort's providers that month). Max-normalization was rejected twice over: the wait-share components are already proportions whose maximum is ~1.0 (printed below), so dividing by the max is the identity and a 0.7 threshold would nearly never fire; and any whole-panel statistic would read the sealed year. Pooled-over-time ranks were also rejected: post-Covid pressure is ~81× the pre-Covid level, so pooled ranks would measure the era, not the provider — allocation needs *who is under most pressure now, relative to peers*.
2. **The proposed REDISTRIBUTE_OUT rule is dead under persistence** — it requires the *lower* band edge ≥ 1.1 × current, but the lower edge is current − band < current, always. It is documented here instead of coded; an OUT-trigger needs a different definition (supervisor decision).
3. The band is **recomputed, not copied**: calibrated on 6-month persistence errors with targets before Apr 2023, so the simulation window (decisions Apr 2023 onward) never overlaps its own calibration.

In [1]:
import pandas as pd                          # tables
import numpy as np                           # arithmetic and np.select for the decision rule
from pathlib import Path

PROC = Path("../data/processed")
TEST_START = pd.Timestamp("2025-04-01")      # sealed test year - never read below
SIM_START = pd.Timestamp("2023-04-01")       # decisions simulated from this origin month onward

W_GT18, W_GT52, W_OCC = 0.2, 0.6, 0.2        # declared 2026-07-30, pending supervisor ratification
P_HIGH, P_MID = 0.70, 0.40                   # pressure thresholds (weighted within-month percentile)
G_RISK, G_STABLE, OCC_SLACK = 1.20, 1.05, 0.80   # growth-risk, stability, spare-capacity thresholds

panel = pd.read_csv(PROC / "rtt_kh03_canonical_panel.csv", parse_dates=["month"])
dev = panel[panel["month"] < TEST_START]

# cohort: C_110 at public allocatable providers (the pool an allocator can actually move work to)
df = dev[(dev["tfunc_code"] == "C_110") & dev["is_public"] & dev["fl_allocatable"]].copy()
df = df.sort_values(["provider_code", "month"]).reset_index(drop=True)
print(df["provider_code"].nunique(), "providers,", len(df), "provider-months")
print("component maxima (why max-normalization is the identity):",
      df[["pct_gt18", "pct_gt52", "occupancy_ga"]].max().round(3).to_dict())

150 providers, 11373 provider-months
component maxima (why max-normalization is the identity): {'pct_gt18': 1.0, 'pct_gt52': 1.0, 'occupancy_ga': 1.0}


## Pressure score

Each component becomes its percentile rank **within its month** across the cohort (0 = least pressured provider that month, 1 = most). The score is the declared weighted sum, so 0.70 reads as "weighted standing around the top third of peers this month".

In [2]:
# Goal: pressure_score per provider-month from within-month component ranks.
for c in ["pct_gt18", "pct_gt52", "occupancy_ga"]:
    df[f"rank_{c}"] = df.groupby("month")[c].rank(pct=True)

occ_rank = np.where(df["rank_occupancy_ga"].notna(), df["rank_occupancy_ga"], 0.5)
# providers with no KH03 G&A return that month (~1% of public rows, see 06) get the
# neutral mid-rank: unknown occupancy neither raises nor lowers their pressure

df["pressure_score"] = W_GT18 * df["rank_pct_gt18"] + W_GT52 * df["rank_pct_gt52"] + W_OCC * occ_rank
print(df["pressure_score"].describe().round(3))

count    11373.000
mean         0.504
std          0.220
min          0.024
25%          0.344
50%          0.486
75%          0.676
max          0.994
Name: pressure_score, dtype: float64


## Forecast layer: 6-month persistence + calibrated band

Forecast = current level; band = 90th percentile of |actual 6-month change| over calibration rows (targets before Apr 2023). Simulated decisions use origins Apr 2023 – Sep 2024, whose 6-month outcomes all fall inside the development window — so every decision can be checked against what actually happened. Origins Oct 2022 – Mar 2023 fall in neither set (their targets straddle the boundary) and act as an unused buffer.

In [3]:
# Goal: the band, then the simulation frame where every row has a known outcome.
g = df.groupby("provider_code")["total_waiting"]
df["actual_6m"] = g.shift(-6)                # the list 6 reported months later
df["target_month"] = df["month"] + pd.DateOffset(months=6)

d = df.dropna(subset=["actual_6m"])          # origins whose outcome falls past Mar 2025 drop out
assert d["target_month"].max() < TEST_START  # a sealed-year outcome would trip this

cal = d[d["target_month"] < SIM_START]       # 6-month persistence errors, targets before Apr 2023
q90 = np.quantile(np.abs(cal["actual_6m"] - cal["total_waiting"]), 0.90)
print(f"6-month 90% band: +/-{q90:,.0f} patients (calibrated on {len(cal)} rows, targets before Apr 2023)")

sim = d[d["month"] >= SIM_START].copy()      # decision months Apr 2023 - Sep 2024
sim["upper_6m"] = sim["total_waiting"] + q90 # persistence forecast = current level, so the band
sim["lower_6m"] = sim["total_waiting"] - q90 # sits symmetrically around it
print(f"simulation: {len(sim)} provider-months, origins {sim['month'].min().date()} -> {sim['month'].max().date()}")

6-month 90% band: +/-1,171 patients (calibrated on 7357 rows, targets before Apr 2023)
simulation: 2337 provider-months, origins 2023-04-01 -> 2024-09-01


## The decision rule

- **EXPAND_CAPACITY** — pressure ≥ 0.70 and the band admits ≥ +20% growth (`upper ≥ 1.20 × current`).
- **REDISTRIBUTE_IN** — pressure in [0.40, 0.70), occupancy < 0.80 (unknown occupancy never counts as slack — a NaN comparison is False), and the band caps growth at +5%.
- **STATUS_QUO** — everything else. REDISTRIBUTE_OUT is omitted as arithmetically dead (see header).

Note a consequence of an *absolute* pooled band on a persistence forecast: `upper ≥ 1.20 × current` reduces to `band ≥ 0.20 × current`, which fires more easily for small waiting lists. This is reported, not hidden — a per-provider or relative band is a candidate refinement for the supervisor discussion.

In [4]:
# Goal: one decision per provider-month, vectorised - no row-wise apply.
high = sim["pressure_score"] >= P_HIGH
mid = (sim["pressure_score"] >= P_MID) & ~high
growth_risk = sim["upper_6m"] >= G_RISK * sim["total_waiting"]
stable = sim["upper_6m"] <= G_STABLE * sim["total_waiting"]
slack = sim["occupancy_ga"] < OCC_SLACK      # NaN occupancy -> False: unknown capacity is never slack

sim["decision"] = np.select([high & growth_risk, mid & stable & slack],
                            ["EXPAND_CAPACITY", "REDISTRIBUTE_IN"], "STATUS_QUO")

n = len(sim)
print(sim["decision"].value_counts().rename("provider-months (of %d)" % n))
exp = sim[sim["decision"] == "EXPAND_CAPACITY"]
print(f"\n{exp['provider_code'].nunique()} providers ever trigger EXPAND; top 10 by trigger months:")
print(exp.groupby("trust_name").size().sort_values(ascending=False).head(10))

decision
STATUS_QUO         2003
EXPAND_CAPACITY     334
Name: provider-months (of 2337), dtype: int64

30 providers ever trigger EXPAND; top 10 by trigger months:
trust_name
BUCKINGHAMSHIRE HEALTHCARE NHS TRUST                     18
CHESTERFIELD ROYAL HOSPITAL NHS FOUNDATION TRUST         18
EAST AND NORTH HERTFORDSHIRE TEACHING NHS TRUST          18
JAMES PAGET UNIVERSITY HOSPITALS NHS FOUNDATION TRUST    18
TORBAY AND SOUTH DEVON NHS FOUNDATION TRUST              18
NORTH CHESHIRE AND MERSEY NHS FOUNDATION TRUST           18
WYE VALLEY NHS TRUST                                     18
THE SHREWSBURY AND TELFORD HOSPITAL NHS TRUST            18
THE PRINCESS ALEXANDRA HOSPITAL NHS TRUST                18
UNIVERSITY HOSPITALS PLYMOUTH NHS TRUST                  18
dtype: int64


## Did the decisions point at the right places?

Every simulated decision has a known 6-month outcome. If the rule is sane, EXPAND months should precede larger actual growth than STATUS_QUO months.

In [5]:
# Goal: realized 6-month change by decision class - the reality check.
sim["realized_change"] = sim["actual_6m"] - sim["total_waiting"]
sim["realized_growth"] = np.where(sim["total_waiting"] > 0,
                                  sim["realized_change"] / sim["total_waiting"], np.nan)
# a zero current list makes growth undefined - those rows keep the absolute change only

chk = sim.groupby("decision").agg(
    n=("realized_change", "size"),
    median_change=("realized_change", "median"),
    median_growth=("realized_growth", "median"),
    grew=("realized_change", lambda s: (s > 0).mean()))
print(chk.round(3))
print("\n'grew' = share of provider-months whose list was larger 6 months later (denominator: n per row)")

                    n  median_change  median_growth   grew
decision                                                  
EXPAND_CAPACITY   334            0.5          0.000  0.500
STATUS_QUO       2003           37.0          0.009  0.535

'grew' = share of provider-months whose list was larger 6 months later (denominator: n per row)


## v2 — Redistribution on within-ICB standing

The v1 reality check above is the design input: growth-risk conditions select nothing (random walk) and absolute-band tests cannot fire, so v2 drops both. Decisions now condition on **current standing among ICB peers** — the pool an allocator can actually move demand within. Declared thresholds (2026-07-30, pending supervisor ratification): **REDISTRIBUTE_OUT** = pressure rank ≥ 0.80 *and* occupancy rank ≥ 0.80 within (month, ICB); **REDISTRIBUTE_IN** = both ranks ≤ 0.30; STATUS_QUO otherwise. Small ICBs make ranks coarse (a 2-provider ICB only has ranks 0.5 and 1.0) — group sizes are printed for context.

In [6]:
# Goal: v2 decisions from within-ICB pressure and occupancy standing.
sim["icb_p_rank"] = sim.groupby(["month", "icb_code"])["pressure_score"].rank(pct=True)
sim["icb_o_rank"] = sim.groupby(["month", "icb_code"])["occupancy_ga"].rank(pct=True)
# a missing ICB code or occupancy leaves NaN ranks; NaN comparisons are False below,
# so such rows fall through to STATUS_QUO
sizes = sim.groupby(["month", "icb_code"]).size()
print("providers per (month, ICB): min %d | median %d | max %d over %d groups"
      % (sizes.min(), sizes.median(), sizes.max(), len(sizes)))

OUT_THR, IN_THR = 0.80, 0.30                 # declared 2026-07-30, pending supervisor ratification
snd = (sim["icb_p_rank"] >= OUT_THR) & (sim["icb_o_rank"] >= OUT_THR)
rcv = (sim["icb_p_rank"] <= IN_THR) & (sim["icb_o_rank"] <= IN_THR)
sim["decision_v2"] = np.select([snd, rcv], ["REDISTRIBUTE_OUT", "REDISTRIBUTE_IN"], "STATUS_QUO")
print(sim["decision_v2"].value_counts().rename("provider-months (of %d)" % len(sim)))

providers per (month, ICB): min 1 | median 3 | max 10 over 648 groups
decision_v2
STATUS_QUO          1775
REDISTRIBUTE_OUT     457
REDISTRIBUTE_IN      105
Name: provider-months (of 2337), dtype: int64


In [7]:
# Goal: can each flagged sender actually find a receiver in its own ICB that month,
# and how do the v2 classes look against their realized 6-month outcomes?
senders = sim[sim["decision_v2"] == "REDISTRIBUTE_OUT"]
recv_slots = sim[sim["decision_v2"] == "REDISTRIBUTE_IN"][["month", "icb_code"]].drop_duplicates()
m = senders.merge(recv_slots, on=["month", "icb_code"], how="left", indicator=True)
print(f"OUT rows with at least one IN receiver in the same ICB-month: "
      f"{(m['_merge'] == 'both').mean():.1%} of {len(senders)}")

icb = sim.groupby("icb_code")["decision_v2"].value_counts().unstack(fill_value=0)
print("\ntop 10 ICBs by REDISTRIBUTE_OUT months:")
print(icb.sort_values("REDISTRIBUTE_OUT", ascending=False).head(10))

chk2 = sim.groupby("decision_v2").agg(n=("realized_change", "size"),
                                      median_change=("realized_change", "median"),
                                      median_pressure=("pressure_score", "median"))
print(chk2.round(3))

OUT rows with at least one IN receiver in the same ICB-month: 13.6% of 457

top 10 ICBs by REDISTRIBUTE_OUT months:
decision_v2  REDISTRIBUTE_IN  REDISTRIBUTE_OUT  STATUS_QUO
icb_code                                                  
QWO                        0                29          61
QRL                       12                24          54
S9B9J                      3                21         102
Z9B2Z                     36                20         111
QR1                        0                18           0
QOP                        0                18          90
QOC                        0                18          36
QGH                        0                18          36
QK1                        0                18           0
QKS                        3                18          69
                     n  median_change  median_pressure
decision_v2                                           
REDISTRIBUTE_IN    105            5.0            0.183
REDISTRIBUT

In [8]:
# Goal: store the simulated decisions so these numbers stay re-readable without a rerun.
out = sim[["month", "provider_code", "trust_name", "icb_code", "total_waiting", "pct_gt18",
           "pct_gt52", "occupancy_ga", "pressure_score", "upper_6m", "lower_6m",
           "decision", "icb_p_rank", "icb_o_rank", "decision_v2", "actual_6m", "realized_change"]]
out.to_csv(PROC / "agent_decisions_h6_c110.csv", index=False)
print(out.shape, "->", PROC / "agent_decisions_h6_c110.csv")

(2337, 17)

 -> ..\data\processed\agent_decisions_h6_c110.csv
